# Экспоненциальный рост

**Цель:** численно решить уравнение $du/dt = \alpha u$ и сравнить результат
с аналитическим решением $u(t)=u_0e^{\alpha t}$.

## Инициализация проекта

In [1]:
using DrWatson
@quickactivate "project"

ENV["GKSwstype"] = "100"

using CSV
using DataFrames
using DifferentialEquations
using JLD2
using Plots

include(srcdir("exponential_growth.jl"))

script_name = "01_exponential_growth"
mkpath(datadir(script_name))
mkpath(plotsdir(script_name))

"/workspace/lab01/plots/01_exponential_growth"

## Параметры модели

Начальная величина равна единице, скорость роста равна `0.3`, а наблюдение
продолжается десять единиц времени.

In [2]:
u0 = 1.0
alpha = 0.3
tspan = (0.0, 10.0)
saveat = 0.1

0.1

## Численное решение

In [3]:
problem = ODEProblem(exponential_growth!, [u0], tspan, alpha)
solution = solve(problem, Tsit5(); saveat=saveat)

results = DataFrame(
    time=solution.t,
    numerical=first.(solution.u),
    analytical=analytic_solution.(u0, alpha, solution.t),
)
results.error = abs.(results.numerical .- results.analytical)

101-element Vector{Float64}:
 0.0
 3.070876886113183e-12
 8.607607737687317e-9
 1.917419933938902e-8
 8.959291530885594e-9
 9.983218030029661e-9
 1.5422747079441024e-8
 2.769486417975031e-9
 3.7628629989683304e-8
 1.5665316110968774e-7
 2.1125137883437617e-7
 1.4489118749239083e-7
 1.806096405765345e-8
 ⋮
 7.695972799837136e-5
 0.00010421983701291992
 0.00013105340194208281
 0.00015424410731768035
 0.0001708348118150127
 0.00017850583934730935
 0.00017596479069581505
 0.00016334822689145767
 0.00014263558540150711
 0.00011807570222899244
 9.662632328755194e-5
 8.84070009306015e-5

## Проверка и сохранение результатов

In [4]:
final_population = last(results.numerical)
expected_population = last(results.analytical)
relative_error = abs(final_population - expected_population) / expected_population
time_to_double = doubling_time(alpha)

CSV.write(datadir(script_name, "trajectory.csv"), results)
jldsave(
    datadir(script_name, "all_results.jld2");
    parameters=(u0=u0, alpha=alpha, tspan=tspan, saveat=saveat),
    results=results,
    relative_error=relative_error,
)

trajectory_plot = plot(
    results.time,
    results.numerical;
    label="Численное решение",
    xlabel="Время, t",
    ylabel="Величина, u(t)",
    title="Экспоненциальный рост (α = $(alpha))",
    linewidth=3,
    legend=:topleft,
    size=(900, 540),
    dpi=150,
)
plot!(
    trajectory_plot,
    results.time,
    results.analytical;
    label="Аналитическое решение",
    linestyle=:dash,
    linewidth=2,
)
savefig(trajectory_plot, plotsdir(script_name, "exponential_growth.png"))

println("Экспоненциальный рост: одиночный эксперимент")
println("  u(0) = $(u0), α = $(alpha), t ∈ $(tspan)")
println("  u(10) численно      = $(round(final_population; digits=6))")
println("  u(10) аналитически  = $(round(expected_population; digits=6))")
println("  относительная ошибка = $(round(relative_error; sigdigits=4))")
println("  время удвоения       = $(round(time_to_double; digits=4))")
println("Результаты: data/$(script_name), plots/$(script_name)")

Экспоненциальный рост: одиночный эксперимент


  u(0) = 1.0, α = 0.3, t ∈ (0.0, 10.0)
  u(10) численно      = 20.085449
  u(10) аналитически  = 20.085537
  относительная ошибка = 4.402e-6
  время удвоения       = 2.3105
Результаты: data/01_exponential_growth, plots/01_exponential_growth


Полученная численная траектория совпадает с аналитической с малой
относительной ошибкой, что подтверждает корректность реализации.